In [ ]:
# Cell 1 — Clone repository and configure Python path
import os, sys

REPO_DIR = '/content/guardian-pixel'

if os.path.isdir(REPO_DIR):
    print('Repo already exists — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/GorohovskiMax/guardian-pixel.git {REPO_DIR}

%cd /content/guardian-pixel
sys.path.insert(0, '/content/guardian-pixel')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 2 — Mount Google Drive and define ArtiFact dataset path
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_ROOT = '/content/drive/MyDrive/ArtiFact'
print(f'Dataset root: {ARTIFACT_ROOT}')

In [ ]:
# Cell 3 — Install missing dependencies
!pip install -q wandb timm albumentations ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
print('All dependencies installed.')

In [ ]:
# Cell 4 — Verify GPU availability and print device info
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {props.name}')
    print(f'VRAM total      : {props.total_memory / 1e9:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. Training will be extremely slow.')

!nvidia-smi

In [ ]:
# Cell 5 — ForensicDetector sanity check
#
# Confirms:
#   1. FSR applied  — stem[0] stride should be (2, 2), not (4, 4)
#   2. Forward pass — output shape must be (1, 7) for the 7-class head

import torch
from layers.forensic import ForensicDetector

CONFIG = 'configs/layer_a.yaml'

model = ForensicDetector.from_config(CONFIG)

print('=== stem after FSR ===')
print(model.model.stem[0])
print()

dummy = torch.randn(1, 3, 200, 200)
model.eval()
with torch.no_grad():
    logits = model(dummy)

assert logits.shape == (1, 7), f'Expected (1, 7), got {tuple(logits.shape)}'
print(f'Forward pass output shape : {tuple(logits.shape)}  OK')

In [ ]:
# Cell 6 — Full training run
#
# Reads all hyperparameters from configs/layer_a.yaml.
# ARTIFACT_ROOT overrides data.root so the Drive path is used.
# W&B run name is read from logging.wandb_run_name in the config.

from layers.training import train

trained_model = train(
    config_path='configs/layer_a.yaml',
    data_root=ARTIFACT_ROOT,
    wandb_project='guardian-pixel',
)

In [ ]:
# Cell 7 — Load best checkpoint and evaluate on the test set

import torch
from layers.forensic import ForensicDetector
from layers.training import evaluate
from utils.dataloader import get_dataloaders

CONFIG     = 'configs/layer_a.yaml'
CHECKPOINT = 'models/layer_a/best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Restore best weights
model = ForensicDetector.from_config(CONFIG).to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(
    f'Loaded checkpoint  epoch={checkpoint["epoch"]}  '
    f'val_bal_acc={checkpoint["best_balanced_accuracy"]:.4f}'
)

# Test set evaluation
loaders = get_dataloaders(CONFIG, root=ARTIFACT_ROOT)
test_metrics = evaluate(model, loaders['test'], device)

print()
print('=== Test Set Results ===')
print(f'  Loss              : {test_metrics["loss"]:.4f}')
print(f'  Accuracy          : {test_metrics["accuracy"]:.4f}')
print(f'  Balanced Accuracy : {test_metrics["balanced_accuracy"]:.4f}')